# Phase 3: Baseline Experiments

This notebook trains the first trustworthy baselines for the weather-aware grape disease project.

Baselines included:

1. Weather-only MLP
2. Image-only MobileNetV2
3. MobileNetV2 + weather concatenation

Expected Phase 2 inputs:

```text
DATASET/docs/phase2_outputs/multimodal_train.csv
DATASET/docs/phase2_outputs/multimodal_val.csv
DATASET/docs/phase2_outputs/multimodal_test.csv
```

Outputs:

```text
DATASET/docs/phase3_outputs/phase3_results.csv
DATASET/docs/phase3_outputs/confusion_matrix_<model>.csv
DATASET/docs/phase3_outputs/best_<model>.pt
```

## 1. Setup

In Colab, mount Google Drive if your project is in Drive, then set `PROJECT_ROOT` to that folder.

In [4]:
from pathlib import Path
import json
import random
import time

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

try:
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"])
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report


# Example for Colab Drive:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = Path("/content/drive/MyDrive")

DATASET_ROOT = PROJECT_ROOT / "DATASET"
PHASE2_DIR = DATASET_ROOT / "docs" / "phase2_outputs"
OUTPUT_DIR = DATASET_ROOT / "docs" / "phase3_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Phase 2 dir:", PHASE2_DIR)
print("Output dir:", OUTPUT_DIR)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Mounted at /content/drive
Project root: /content/drive/MyDrive
Phase 2 dir: /content/drive/MyDrive/DATASET/docs/phase2_outputs
Output dir: /content/drive/MyDrive/DATASET/docs/phase3_outputs
Torch: 2.11.0+cu130
CUDA available: True


## 2. Configuration

These defaults are chosen for Colab free GPU. Increase epochs later only after the full pipeline works.

In [5]:
SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

EPOCHS_WEATHER = 60
EPOCHS_IMAGE = 5
EPOCHS_MULTIMODAL = 5
PATIENCE = 5

LEARNING_RATE_WEATHER = 1e-3
LEARNING_RATE_IMAGE = 3e-4
WEIGHT_DECAY = 1e-4

PRETRAINED = True
FREEZE_BACKBONE = True

RUN_WEATHER_ONLY = True
RUN_IMAGE_ONLY = True
RUN_MULTIMODAL = True

WEATHER_COLUMNS = ["temperature", "humidity", "rainfall"]

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## 3. Load Phase 2 CSVs

In [6]:
train_df = pd.read_csv(PHASE2_DIR / "multimodal_train.csv")
val_df = pd.read_csv(PHASE2_DIR / "multimodal_val.csv")
test_df = pd.read_csv(PHASE2_DIR / "multimodal_test.csv")

all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
class_names = sorted(train_df["label"].unique().tolist())
label_to_idx = {label: idx for idx, label in enumerate(class_names)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}

for frame in [train_df, val_df, test_df]:
    frame["label_idx"] = frame["label"].map(label_to_idx)

print("Classes:", class_names)
print("Rows:", len(train_df), len(val_df), len(test_df))
display(pd.DataFrame({
    "train": train_df["label"].value_counts(),
    "val": val_df["label"].value_counts(),
    "test": test_df["label"].value_counts(),
}).fillna(0).astype(int))

# Leakage sanity check: no split_group should appear in multiple splits.
split_check = all_df.groupby("split_group")["split"].nunique().max()
print("Max splits per split_group:", split_check)
if split_check != 1:
    raise RuntimeError("Leakage issue: at least one split_group appears in multiple splits.")

Classes: ['Black Rot', 'Esca', 'Healthy', 'Leaf Blight']
Rows: 2245 578 654


,train,val,test
label,,,
Black Rot,576,112,120
Esca,535,157,196
Healthy,664,207,238
Leaf Blight,470,102,100


Max splits per split_group: 1


## 4. Weather Normalization

Weather features are normalized using training-set statistics only.

In [7]:
weather_mean = train_df[WEATHER_COLUMNS].mean()
weather_std = train_df[WEATHER_COLUMNS].std().replace(0, 1)

normalization = {
    "weather_columns": WEATHER_COLUMNS,
    "mean": weather_mean.to_dict(),
    "std": weather_std.to_dict(),
    "class_names": class_names,
}
(OUTPUT_DIR / "normalization.json").write_text(json.dumps(normalization, indent=2))

display(pd.DataFrame({"mean": weather_mean, "std": weather_std}))

,mean,std
temperature,27.888136,0.271227
humidity,68.552448,1.164107
rainfall,15.728102,0.387312


## 5. Datasets And Loaders

In [8]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

class GrapeDiseaseDataset(Dataset):
    def __init__(self, frame, dataset_root, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.dataset_root = Path(dataset_root)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image_path = self.dataset_root / row["image_path"]
        image = Image.open(image_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)

        weather = ((row[WEATHER_COLUMNS].astype(float) - weather_mean) / weather_std).values.astype("float32")
        label = int(row["label_idx"])
        return image, torch.tensor(weather, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

train_ds = GrapeDiseaseDataset(train_df, DATASET_ROOT, train_transform)
val_ds = GrapeDiseaseDataset(val_df, DATASET_ROOT, eval_transform)
test_ds = GrapeDiseaseDataset(test_df, DATASET_ROOT, eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

len(train_loader), len(val_loader), len(test_loader)

(71, 19, 21)

## 6. Models

In [9]:
class WeatherMLP(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(len(WEATHER_COLUMNS), 32),
            nn.ReLU(inplace=True),
            nn.Dropout(0.15),
            nn.Linear(32, 32),
            nn.ReLU(inplace=True),
            nn.Linear(32, num_classes),
        )

    def forward(self, images, weather):
        return self.net(weather)


class ImageOnlyMobileNetV2(nn.Module):
    def __init__(self, num_classes, pretrained=True, freeze_backbone=True):
        super().__init__()
        weights = models.MobileNet_V2_Weights.DEFAULT if pretrained else None
        self.backbone = models.mobilenet_v2(weights=weights)
        if freeze_backbone:
            for param in self.backbone.features.parameters():
                param.requires_grad = False
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier[1] = nn.Linear(in_features, num_classes)

    def forward(self, images, weather):
        return self.backbone(images)


class MobileNetV2WeatherConcat(nn.Module):
    def __init__(self, num_classes, pretrained=True, freeze_backbone=True):
        super().__init__()
        weights = models.MobileNet_V2_Weights.DEFAULT if pretrained else None
        base = models.mobilenet_v2(weights=weights)
        self.features = base.features
        if freeze_backbone:
            for param in self.features.parameters():
                param.requires_grad = False

        self.weather_encoder = nn.Sequential(
            nn.Linear(len(WEATHER_COLUMNS), 16),
            nn.ReLU(inplace=True),
            nn.Linear(16, 32),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(1280 + 32, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes),
        )

    def forward(self, images, weather):
        x = self.features(images)
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)
        w = self.weather_encoder(weather)
        return self.classifier(torch.cat([x, w], dim=1))

## 7. Training And Evaluation Utilities

In [10]:
def make_class_weights(frame):
    counts = frame["label_idx"].value_counts().sort_index().values.astype("float32")
    weights = counts.sum() / (len(counts) * counts)
    return torch.tensor(weights, dtype=torch.float32, device=device)

class_weights = make_class_weights(train_df)
class_weights

tensor([0.9744, 1.0491, 0.8453, 1.1941], device='cuda:0')

In [11]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss = 0.0
    all_preds = []
    all_targets = []

    for images, weather, targets in loader:
        images = images.to(device, non_blocking=True)
        weather = weather.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with torch.set_grad_enabled(is_train):
            logits = model(images, weather)
            loss = criterion(logits, targets)

            if is_train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * targets.size(0)
        all_preds.extend(logits.argmax(dim=1).detach().cpu().numpy().tolist())
        all_targets.extend(targets.detach().cpu().numpy().tolist())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_targets, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_targets, all_preds, average="macro", zero_division=0
    )
    return {"loss": avg_loss, "accuracy": acc, "precision": precision, "recall": recall, "macro_f1": f1}


def fit_model(model_name, model, epochs, learning_rate):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=learning_rate, weight_decay=WEIGHT_DECAY)

    best_val_f1 = -1
    best_path = OUTPUT_DIR / f"best_{model_name}.pt"
    history = []
    bad_epochs = 0
    start = time.time()

    for epoch in range(1, epochs + 1):
        train_metrics = run_epoch(model, train_loader, criterion, optimizer)
        val_metrics = run_epoch(model, val_loader, criterion)
        row = {"model": model_name, "epoch": epoch, **{f"train_{k}": v for k, v in train_metrics.items()}, **{f"val_{k}": v for k, v in val_metrics.items()}}
        history.append(row)
        print(f"{model_name} epoch {epoch:02d}: train_f1={train_metrics['macro_f1']:.4f} val_f1={val_metrics['macro_f1']:.4f} val_acc={val_metrics['accuracy']:.4f}")

        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1 = val_metrics["macro_f1"]
            torch.save(model.state_dict(), best_path)
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print("Early stopping")
                break

    model.load_state_dict(torch.load(best_path, map_location=device))
    test_metrics, y_true, y_pred = evaluate_model(model, test_loader)
    elapsed = time.time() - start
    print(f"{model_name} test: acc={test_metrics['accuracy']:.4f} macro_f1={test_metrics['macro_f1']:.4f} time={elapsed/60:.1f} min")

    pd.DataFrame(history).to_csv(OUTPUT_DIR / f"history_{model_name}.csv", index=False)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(OUTPUT_DIR / f"confusion_matrix_{model_name}.csv")
    report = classification_report(y_true, y_pred, target_names=class_names, zero_division=0, output_dict=True)
    pd.DataFrame(report).transpose().to_csv(OUTPUT_DIR / f"classification_report_{model_name}.csv")

    return {"model": model_name, **test_metrics, "best_val_macro_f1": best_val_f1, "minutes": elapsed / 60}


@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    all_preds = []
    all_targets = []
    for images, weather, targets in loader:
        images = images.to(device, non_blocking=True)
        weather = weather.to(device, non_blocking=True)
        logits = model(images, weather)
        all_preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
        all_targets.extend(targets.numpy().tolist())

    acc = accuracy_score(all_targets, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_targets, all_preds, average="macro", zero_division=0
    )
    return {"accuracy": acc, "precision": precision, "recall": recall, "macro_f1": f1}, all_targets, all_preds

## 8. Run Baselines

In [12]:
results = []

if RUN_WEATHER_ONLY:
    seed_everything(SEED)
    results.append(fit_model(
        "weather_mlp",
        WeatherMLP(num_classes=len(class_names)),
        EPOCHS_WEATHER,
        LEARNING_RATE_WEATHER,
    ))

if RUN_IMAGE_ONLY:
    seed_everything(SEED)
    results.append(fit_model(
        "mobilenetv2_image_only",
        ImageOnlyMobileNetV2(len(class_names), pretrained=PRETRAINED, freeze_backbone=FREEZE_BACKBONE),
        EPOCHS_IMAGE,
        LEARNING_RATE_IMAGE,
    ))

if RUN_MULTIMODAL:
    seed_everything(SEED)
    results.append(fit_model(
        "mobilenetv2_weather_concat",
        MobileNetV2WeatherConcat(len(class_names), pretrained=PRETRAINED, freeze_backbone=FREEZE_BACKBONE),
        EPOCHS_MULTIMODAL,
        LEARNING_RATE_IMAGE,
    ))

results_df = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
results_path = OUTPUT_DIR / "phase3_results.csv"
results_df.to_csv(results_path, index=False)
display(results_df)
print("Saved:", results_path)

weather_mlp epoch 01: train_f1=0.1481 val_f1=0.1376 val_acc=0.3581
weather_mlp epoch 02: train_f1=0.2378 val_f1=0.1376 val_acc=0.3581
weather_mlp epoch 03: train_f1=0.2302 val_f1=0.1376 val_acc=0.3581
weather_mlp epoch 04: train_f1=0.2199 val_f1=0.1376 val_acc=0.3581
weather_mlp epoch 05: train_f1=0.2174 val_f1=0.0700 val_acc=0.1592
weather_mlp epoch 06: train_f1=0.2437 val_f1=0.1376 val_acc=0.3581
Early stopping
weather_mlp test: acc=0.3471 macro_f1=0.2008 time=15.6 min
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 143MB/s]


mobilenetv2_image_only epoch 01: train_f1=0.5919 val_f1=0.7632 val_acc=0.7889
mobilenetv2_image_only epoch 02: train_f1=0.7760 val_f1=0.7540 val_acc=0.8097
mobilenetv2_image_only epoch 03: train_f1=0.7926 val_f1=0.8154 val_acc=0.8408
mobilenetv2_image_only epoch 04: train_f1=0.7896 val_f1=0.8128 val_acc=0.8443
mobilenetv2_image_only epoch 05: train_f1=0.8129 val_f1=0.8295 val_acc=0.8512
mobilenetv2_image_only test: acc=0.8043 macro_f1=0.7570 time=4.6 min
mobilenetv2_weather_concat epoch 01: train_f1=0.7007 val_f1=0.8304 val_acc=0.8478
mobilenetv2_weather_concat epoch 02: train_f1=0.8056 val_f1=0.8131 val_acc=0.8478
mobilenetv2_weather_concat epoch 03: train_f1=0.8352 val_f1=0.8349 val_acc=0.8547
mobilenetv2_weather_concat epoch 04: train_f1=0.8234 val_f1=0.8191 val_acc=0.8616
mobilenetv2_weather_concat epoch 05: train_f1=0.8479 val_f1=0.8281 val_acc=0.8599
mobilenetv2_weather_concat test: acc=0.8028 macro_f1=0.7640 time=4.6 min


,model,accuracy,precision,recall,macro_f1,best_val_macro_f1,minutes
2,mobilenetv2_weather_concat,0.802752,0.788898,0.758191,0.763960,0.834925,4.617011
1,mobilenetv2_image_only,0.804281,0.780782,0.747672,0.756951,0.829502,4.640751
0,weather_mlp,0.347095,0.178502,0.253076,0.200815,0.137633,15.572044


Saved: /content/drive/MyDrive/DATASET/docs/phase3_outputs/phase3_results.csv


## 9. Phase 3 Interpretation Checklist

After the run, check these before moving to Phase 4:

- Weather-only MLP should not be suspiciously strong. If it is strong, weather or split grouping may be leaking class information.
- Image-only MobileNetV2 is the first visual baseline.
- MobileNetV2 + weather concatenation should be compared against image-only MobileNetV2.
- Use `classification_report_*.csv` and `confusion_matrix_*.csv` for class-wise analysis.
- These are not final paper numbers yet. Final numbers need stronger models and repeated seeds.